# Packages Import

In [1]:
import requests
from bs4 import BeautifulSoup

# Ceneo Scraper

1.  Provide url of the products opinions page

In [2]:

product_code = "173179719"

url = f"https://www.ceneo.pl/{product_code}#tab=reviews"

2.Send the request to provided url

In [3]:
response = requests.get(url)
print(response.status_code)

200


3. Fetch the product name

In [4]:
page_dom = BeautifulSoup(response.text,'html.parser')
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [5]:
product_name = page_dom.select_one('h1').get_text()
print(product_name)
print(type(product_name))

Konsola Sony PlayStation 5 Pro
<class 'str'>


4. Fetch all opinions from the webpage

In [12]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


5. Parse opinion to extract required data

In [17]:
OPINION_CONTAINER = "div.js_product-review:not(.user-post--highlight)"

def get_text_safe(el):
    return el.get_text(strip=True) if el else None

def get_attr_safe(el, attr):
    return el.get(attr) if el else None

SELECTORS = {
    "opinion_id": lambda op: op.get("data-entry-id"),

    "author_recommendation": lambda op: get_text_safe(
        op.select_one("span.user-post__author-recomendation > em")
    ),

    "score_stars": lambda op: get_text_safe(
        op.select_one("span.user-post__score-count")
    ),

    "content": lambda op: get_text_safe(
        op.select_one("div.user-post__text")
    ),

    "pros": lambda op: [
        el.get_text(strip=True)
        for el in op.select("div.review-feature__item--positive")
    ],

    "cons": lambda op: [
        el.get_text(strip=True)
        for el in op.select("div.review-feature__item--negative")
    ],

    "helpful_count": lambda op: get_attr_safe(
        op.select_one("button.vote-yes"), "data-total-vote"
    ),

    "unhelpful_count": lambda op: get_attr_safe(
        op.select_one("button.vote-no"), "data-total-vote"
    ),

    "publishing_date": lambda op: get_text_safe(
        op.select_one("span.user-post__published > time:nth-of-type(1)")
    ),

    "purchase_date": lambda op: get_text_safe(
        op.select_one("span.user-post__published > time:nth-of-type(2)")
    ),
}

opinions = page_dom.select(OPINION_CONTAINER)

for op in opinions:
    data = {key: func(op) for key, func in SELECTORS.items()}
    print(data)

{'opinion_id': '19859983', 'author_recommendation': 'Polecam', 'score_stars': '5/5', 'content': 'Bardzo dobrza konsola', 'pros': ['cena', 'grywalność', 'jakość', 'możliwości'], 'cons': [], 'helpful_count': '0', 'unhelpful_count': '1', 'publishing_date': '8 miesięcy temu,', 'purchase_date': 'po 2 tygodniach'}
{'opinion_id': '20129492', 'author_recommendation': 'Polecam', 'score_stars': '4,5/5', 'content': 'Konsola sprawuje się doskonale. Chociaż wciąż czekam na gry, które w pełni wykorzystają moc PRO, co uzasadni jej koszt ;). Póki co to sprzęt wyłącznie dla entuzjastów. Niedzielnemu graczowi spokojnie wystarczy wersja podstawowa.', 'pros': ['grywalność', 'jakość', 'możliwości'], 'cons': ['cena'], 'helpful_count': '0', 'unhelpful_count': '0', 'publishing_date': '5 miesięcy temu,', 'purchase_date': 'po 2 tygodniach'}
{'opinion_id': '20192537', 'author_recommendation': 'Polecam', 'score_stars': '5/5', 'content': 'Rewelacyjny sprzęt, cicho pracuje, efekty graficzne w połączeniu z telewizor